
# 三栏轨迹演示（Python Notebook 版）

此 Notebook 复刻了你上传的 HTML Demo 的核心交互：
- 左：目标帧（`target.png`）
- 中：3D 轨迹按帧播放（顺序播放多条轨迹）
- 右：导航画面（我们的方法 vs. Baseline），并在每条轨迹播放完毕后展示误差数字 3 秒

### 目录结构（与原 HTML 一致）
```
.
├─ target.png
├─ run_000/
│  ├─ metadata.json   # 包含 frames 列表（每帧 pose 与 png 相对路径）
│  └─ frames/
│     ├─ frame_000.png
│     ├─ frame_001.png
│     └─ ...
├─ run_001/
│  └─ ...
├─ run_002/
│  └─ ...
└─ editor-2/
   ├─ run_006/frames/...
   ├─ run_001/frames/...
   └─ run_003/frames/...
```
> **如果上述真实数据不存在**，本 Notebook 会自动**生成一个可运行的仿真数据集**（三条轨迹与对应的帧图像与 `metadata.json`），以便你直接运行和查看效果。

### 使用方法
1. 运行 **所有单元格**；若没有你自己的数据，脚本会在当前目录下生成演示数据。
2. 点击“**重新播放**”按钮即可顺序播放 `run_000 → run_001 → run_002`。  
3. 右侧会同步显示 **Method** 与 **Baseline** 的当前帧画面，并在每条轨迹播放完后显示误差数值 3 秒。


In [16]:
# ======== Editor 结构可视化：工具函数 ========
# 目录结构：
# <BASE_DIR>/editor/run_012/
#   ├─ frames/{init.png, step_000.png, ..., goal.png}
#   ├─ metadata.json
#   └─ candidates/
#       ├─ cand_000/{frames/... , metadata.json, deltas.npy}
#       ├─ cand_001/...
#       └─ summary.json  (可选)

import os, json, io, time, glob, re
from pathlib import Path
from typing import List, Dict, Tuple, Optional

from PIL import Image, ImageDraw, ImageFont
import numpy as np
import ipywidgets as W
from IPython.display import display, clear_output

def _read_json(p: Path) -> Optional[dict]:
    try:
        return json.loads(p.read_text('utf-8'))
    except Exception:
        return None

def _png_bytes(p: Path) -> bytes:
    if not p.exists():
        return b''
    with open(p, 'rb') as f:
        return f.read()

def _text_width(draw: ImageDraw.ImageDraw, text: str, font=None) -> int:
    # 兼容旧版 Pillow：优先 textlength，退化到 textbbox
    try:
        return int(draw.textlength(text, font=font))
    except Exception:
        box = draw.textbbox((0,0), text, font=font)
        return int(box[2] - box[0])

def _overlay_box_text(im: Image.Image, text: str):
    im = im.copy().convert("RGB")
    draw = ImageDraw.Draw(im)
    try:
        font = ImageFont.load_default()
    except Exception:
        font = None
    pad = 8
    w = _text_width(draw, text, font)
    h = 14
    x = (im.width - (w + 2*pad)) // 2
    y = 6
    # 半透明黑底（简单起见用不透明，Jupyter 显示效果更稳定）
    draw.rectangle([x, y, x+w+2*pad, y+h+2*pad], fill=(0,0,0))
    draw.text((x+pad, y+pad), text, fill=(255,255,255), font=font)
    return im

def _compose_side_by_side(a: Image.Image, b: Image.Image) -> Image.Image:
    w = a.width + b.width
    h = max(a.height, b.height)
    canvas = Image.new('RGB', (w, h), (246,247,249))
    canvas.paste(a, (0,0))
    canvas.paste(b, (a.width, 0))
    return canvas

def _scan_runs(base_dir: Path) -> List[str]:
    editor = base_dir / "editor"
    if not editor.exists():
        return []
    runs = sorted([p.name for p in editor.iterdir() if p.is_dir() and re.match(r"run_\d{3}$", p.name)])
    return runs

def _list_candidates(run_dir: Path) -> List[str]:
    cdir = run_dir / "candidates"
    if not cdir.exists():
        return []
    # 优先按 summary.json rank
    summary = _read_json(cdir / "summary.json")
    if summary and "items" in summary:
        ids = [f"cand_{it.get('rank', 10**9):03d}" for it in summary["items"]]
        ids = [cid for cid in ids if (cdir / cid).exists()]
        if ids:
            return sorted(set(ids))
    # 回退：cand_XXX 目录名排序
    cands = sorted([p.name for p in cdir.iterdir() if p.is_dir() and re.match(r"cand_\d{3}$", p.name)])
    return cands

def _load_meta_and_frames(run_dir: Path) -> Tuple[dict, List[dict]]:
    meta = _read_json(run_dir / "metadata.json") or {}
    frames = meta.get("frames", [])
    frames = sorted(frames, key=lambda x: x.get("frame", 0))
    return meta, frames

def _goal_path(run_dir: Path) -> Path:
    return run_dir / "frames" / "goal.png"

In [17]:
# ======== Editor 结构可视化：交互式 UI ========

BASE_DIR = W.Text(
    value='/data1/tpz/nwm-main/results/nwm_cdit_airvln_16/airvln_16/CEM_N10_K10_RS1_rep3_OPT1/',  # 在这里填你的输出根目录（包含 editor/ 的那一层）
    description='BASE_DIR:',
    layout=W.Layout(width='60%')
)
BTN_SCAN = W.Button(description='扫描 runs', button_style='info')
RUN = W.Dropdown(options=[], description='run_id:')
VIEW = W.ToggleButtons(options=[('final','final'), ('candidate','candidate')],
                       value='final', description='视图:')
CAND = W.Dropdown(options=[], description='candidate:')
FPS = W.IntSlider(value=2, min=1, max=30, step=1, description='FPS:')
BTN_PLAY = W.Button(description='播放', button_style='success')
BTN_STOP = W.Button(description='停止', button_style='warning')
LBL = W.Label('就绪')
IMG = W.Image(format='png')
OUT = W.Output()

ui_top = W.HBox([BASE_DIR, BTN_SCAN, LBL])
ui_mid = W.HBox([RUN, VIEW, CAND, FPS, BTN_PLAY, BTN_STOP])
ui = W.VBox([ui_top, ui_mid, IMG, OUT])
display(ui)

state = {"playing": False}

def _refresh_candidates(*args):
    base = Path(BASE_DIR.value).expanduser().resolve()
    run_id = RUN.value
    if not run_id:
        CAND.options = []
        return
    run_dir = base / "editor" / run_id
    cands = _list_candidates(run_dir)
    CAND.options = cands
    if cands:
        CAND.value = cands[0]
    # 只有 candidate 视图时显示候选下拉
    CAND.layout.display = 'none' if VIEW.value == 'final' else 'flex'

def _refresh_runs(_=None):
    base = Path(BASE_DIR.value).expanduser().resolve()
    runs = _scan_runs(base)
    if not runs:
        RUN.options = []
        LBL.value = f"未找到 {base}/editor/run_XXX/"
        IMG.value = b''
        return
    RUN.options = runs
    RUN.value = runs[0]
    LBL.value = f"发现 {len(runs)} 个 run"
    _refresh_candidates()
    _render_once()

def _render_once():
    base = Path(BASE_DIR.value).expanduser().resolve()
    if not RUN.value:
        return
    run_dir = base / "editor" / RUN.value
    if VIEW.value == 'candidate':
        run_dir = run_dir / "candidates" / CAND.value

    meta, frames = _load_meta_and_frames(run_dir)
    fps = int(meta.get("fps", FPS.value) or FPS.value)

    if not frames:
        LBL.value = "metadata.json 中无 frames"
        IMG.value = b''
        return

    # 预览第一帧（左：当前帧，右：goal.png）
    f = frames[0]
    png = run_dir / f.get("png", "")
    cur = Image.open(png).convert('RGB') if png.exists() else Image.new('RGB', (224,224), (220,220,220))
    if "loss" in f:
        cur = _overlay_box_text(cur, f"loss={f['loss']:.4f}")
    goal = _goal_path(run_dir)
    if goal.exists():
        goal_im = Image.open(goal).convert('RGB')
        cur = _compose_side_by_side(cur, goal_im)
    bio = io.BytesIO(); cur.save(bio, format='PNG'); IMG.value = bio.getvalue()
    LBL.value = f"预览：{RUN.value} / {VIEW.value}{' / ' + CAND.value if VIEW.value=='candidate' else ''} | FPS={fps}"

def _play(_=None):
    state["playing"] = True
    base = Path(BASE_DIR.value).expanduser().resolve()
    if not RUN.value:
        return
    run_dir = base / "editor" / RUN.value
    if VIEW.value == 'candidate':
        run_dir = run_dir / "candidates" / CAND.value

    meta, frames = _load_meta_and_frames(run_dir)
    fps = int(meta.get("fps", FPS.value) or FPS.value)

    for i, f in enumerate(frames):
        if not state["playing"]:
            break
        png = run_dir / f.get("png","")
        cur = Image.open(png).convert('RGB') if png.exists() else Image.new('RGB',(224,224),(220,220,220))
        if "loss" in f:
            cur = _overlay_box_text(cur, f"loss={f['loss']:.4f}")
        goal = _goal_path(run_dir)
        if goal.exists():
            goal_im = Image.open(goal).convert('RGB')
            cur = _compose_side_by_side(cur, goal_im)
        bio = io.BytesIO(); cur.save(bio, format='PNG'); IMG.value = bio.getvalue()
        LBL.value = f"播放：{RUN.value} / {VIEW.value}{' / ' + CAND.value if VIEW.value=='candidate' else ''} | frame {i+1}/{len(frames)} | FPS={fps}"
        time.sleep(1.0/max(1,fps))
    state["playing"] = False
    LBL.value = "播放结束"

def _stop(_=None):
    state["playing"] = False
    LBL.value = "已停止"

# 事件绑定
BTN_SCAN.on_click(_refresh_runs)
RUN.observe(lambda _: _render_once(), names='value')
VIEW.observe(_refresh_candidates, names='value')
CAND.observe(lambda _: _render_once(), names='value')
FPS.observe(lambda _: _render_once(), names='value')
BTN_PLAY.on_click(_play)
BTN_STOP.on_click(_stop)

# 初次尝试扫描并预览
_refresh_runs()

In [18]:
# ======== 3D 轨迹可视化（GT / Final / Candidate） ========
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import ipywidgets as W
from IPython.display import display, clear_output

# 控件：可选 GT 源、是否画所有候选、是否画 final/candidate/gt
GT_FILE = W.Text(value='', description='GT .npy(可选):', layout=W.Layout(width='60%'))
IDX_IN_GT = W.IntText(value=-1, description='GT行索引:', tooltip='当GT文件是[N,T,4]时, 默认用当前run_index; 设为>=0可手动指定行')
SHOW_GT = W.Checkbox(value=True, description='显示 GT')
SHOW_FINAL = W.Checkbox(value=True, description='显示 Final')
SHOW_CAND = W.Checkbox(value=True, description='显示 Candidate')
ALL_CANDS = W.Checkbox(value=False, description='所有 Candidate')

BTN_PLOT3D = W.Button(description='绘制3D轨迹', button_style='success')
LBL3D = W.Label('就绪')
OUT3D = W.Output()

ui3_top = W.HBox([GT_FILE, IDX_IN_GT])
ui3_mid = W.HBox([SHOW_GT, SHOW_FINAL, SHOW_CAND, ALL_CANDS, BTN_PLOT3D, LBL3D])
ui3 = W.VBox([ui3_top, ui3_mid, OUT3D])
display(ui3)

def _deltas_from_metadata(run_dir: Path):
    """从 metadata.json 的 frames[].delta 读取并拼出 [T,4]；若缺失返回 None"""
    meta, frames = _load_meta_and_frames(run_dir)
    deltas = []
    for f in frames:
        d = f.get('delta')
        if not d:
            continue
        dx = d.get('dx', 0.0); dy = d.get('dy', 0.0); dz = d.get('dz', 0.0); dth = d.get('dtheta', 0.0)
        # INIT 通常是全0，可忽略；但即便包含也问题不大
        if dx==dy==dz==dth==0.0 and f.get('action','INIT')=='INIT':
            continue
        deltas.append([dx, dy, dz, dth])
    if len(deltas)==0:
        return None
    return np.array(deltas, dtype=np.float32)

def _deltas_from_candidate(run_dir: Path):
    """优先 deltas.npy；没有则从 metadata.json 读取"""
    npy = run_dir / 'deltas.npy'
    if npy.exists():
        arr = np.load(npy)
        # 兼容 (1,T,4)
        if arr.ndim == 3 and arr.shape[0] == 1:
            arr = arr[0]
        return arr.astype(np.float32)
    return _deltas_from_metadata(run_dir)

def _deltas_to_xyz(deltas: np.ndarray) -> np.ndarray:
    """[T,4] -> [T+1,3]，从(0,0,0)起累计 xyz"""
    xyz = np.zeros((1,3), dtype=np.float32)
    if deltas is not None and deltas.size > 0:
        steps = np.cumsum(deltas[:, :3], axis=0)  # 只累计 xyz
        xyz = np.vstack([xyz, steps])
    return xyz

def _load_gt_xyz(base: Path, run_id: str, gt_file: str, idx_in_gt: int) -> np.ndarray:
    """从可选的 GT .npy 读取 [T,4] 或 [N,T,4]，转为 xyz；找不到则返回 None"""
    if not gt_file:
        return None
    p = Path(gt_file).expanduser().resolve()
    if not p.exists():
        LBL3D.value = f"GT文件不存在: {p}"
        return None
    arr = np.load(p, allow_pickle=True)
    # 推断当前 run_index
    # run_id 形如 run_012 → 12
    try:
        sid = int(run_id.split('_')[-1])
    except Exception:
        sid = 0
    if idx_in_gt >= 0:
        sel = idx_in_gt
    else:
        sel = sid

    if arr.ndim == 2:           # [T,4] or [T,3]
        deltas = arr
    elif arr.ndim == 3:         # [N,T,4]
        if sel < arr.shape[0]:
            deltas = arr[sel]
        else:
            LBL3D.value = f"GT行索引超界(想要{sel}, 但N={arr.shape[0]}), 用第0行"
            deltas = arr[0]
    else:
        LBL3D.value = f"无法识别GT数组维度: {arr.shape}"
        return None

    # 若只有3列，也按 xyz 解释
    if deltas.shape[1] >= 3:
        deltas = deltas[:, :4] if deltas.shape[1] >= 4 else np.hstack([deltas[:, :3], np.zeros((deltas.shape[0],1), dtype=deltas.dtype)])
        return _deltas_to_xyz(deltas.astype(np.float32))
    return None

def _gather_all_candidates_xyz(run_dir: Path) -> list:
    """返回 [(cand_name, xyz), ...]"""
    ret = []
    cdir = run_dir / 'candidates'
    if not cdir.exists():
        return ret
    # 优先按 summary.json 的 rank 排序
    cands = _list_candidates(run_dir)
    for cid in cands:
        cpath = cdir / cid
        deltas = _deltas_from_candidate(cpath)
        if deltas is None:
            continue
        xyz = _deltas_to_xyz(deltas)
        ret.append((cid, xyz))
    return ret

def _plot_equal_3d(ax, curves, labels, styles=None):
    """curves: list of (N,3); labels对齐。自动设同尺度坐标轴"""
    mins = np.array([np.inf, np.inf, np.inf], dtype=np.float64)
    maxs = -mins
    for xyz in curves:
        if xyz is None or len(xyz)==0: 
            continue
        mins = np.minimum(mins, xyz.min(axis=0))
        maxs = np.maximum(maxs, xyz.max(axis=0))
    # 容错：若全 0
    if not np.all(np.isfinite(mins)) or not np.all(np.isfinite(maxs)):
        mins = np.array([-1,-1,-1], dtype=np.float64)
        maxs = np.array([ 1, 1, 1], dtype=np.float64)
    span = max(1e-6, float(np.max(maxs - mins)))
    center = (mins + maxs) / 2.0
    lo = center - span/2
    hi = center + span/2

    for i, xyz in enumerate(curves):
        if xyz is None or len(xyz)==0: 
            continue
        if styles and i < len(styles):
            ax.plot(xyz[:,0], xyz[:,1], xyz[:,2], styles[i], label=labels[i])
        else:
            ax.plot(xyz[:,0], xyz[:,1], xyz[:,2], label=labels[i])

    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_xlim(lo[0], hi[0]); ax.set_ylim(lo[1], hi[1]); ax.set_zlim(lo[2], hi[2])
    ax.legend(loc='best'); ax.grid(True)

def _do_plot3d(_=None):
    base = Path(BASE_DIR.value).expanduser().resolve()
    run_id = RUN.value
    if not run_id:
        LBL3D.value = "请选择 run_id"
        return

    # Final 轨迹（来自 editor/run_xxx/metadata.json 的 delta）
    final_dir = base / 'editor' / run_id
    final_deltas = _deltas_from_metadata(final_dir)
    final_xyz = _deltas_to_xyz(final_deltas) if final_deltas is not None else None

    # Candidate 轨迹
    cand_xyz_list = []
    if SHOW_CAND.value:
        if ALL_CANDS.value:
            cand_xyz_list = _gather_all_candidates_xyz(final_dir)
        else:
            if CAND.options and CAND.value:
                cdir = final_dir / 'candidates' / CAND.value
                cd = _deltas_from_candidate(cdir)
                if cd is not None:
                    cand_xyz_list = [(CAND.value, _deltas_to_xyz(cd))]

    # GT 轨迹（可选）
    gt_xyz = _load_gt_xyz(base, run_id, GT_FILE.value, IDX_IN_GT.value) if SHOW_GT.value else None

    with OUT3D:
        clear_output(wait=True)
        fig = plt.figure(figsize=(7.5, 6))
        ax = fig.add_subplot(111, projection='3d')

        curves, labels, styles = [], [], []

        if SHOW_GT.value and gt_xyz is not None:
            curves.append(gt_xyz); labels.append('GT'); styles.append('-')     # 线型可按需调整

        if SHOW_FINAL.value and final_xyz is not None:
            curves.append(final_xyz); labels.append('Final'); styles.append('-')

        if SHOW_CAND.value and cand_xyz_list:
            for name, xyz in cand_xyz_list:
                curves.append(xyz); labels.append(name); styles.append('--')

        if not curves:
            LBL3D.value = "没有可画的轨迹（检查 metadata.json 是否含 delta 或提供 GT 文件）"
            plt.close(fig)
            return

        _plot_equal_3d(ax, curves, labels, styles)
        plt.show()
        LBL3D.value = f"绘制完成：{', '.join(labels)}"

BTN_PLOT3D.on_click(_do_plot3d)